[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Preprocessing &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
TEXT = ["terrain", "owner"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Fit a `StandardScaler` on the training rows of `elevation_m` alone, and print the scaled values of the first three test stations.

In [2]:
snow = network["snow_on_may_1"]
train_rows, test_rows = train_test_split(network.index, test_size=0.25, random_state=0,
                                         stratify=snow)

height_scaler = StandardScaler().fit(network.loc[train_rows, ["elevation_m"]])
print(height_scaler.transform(network.loc[test_rows[:3], ["elevation_m"]]).round(2))

[[-1.38]
 [-0.9 ]
 [-1.24]]


The first two lines repeat the worked example's split, since this notebook runs on its own. Double brackets keep one column as a table, which the scaler needs.

**2.** Fit a `LogisticRegression` on the scaled training columns with `C=0.01`, a hundred times the default penalty on large coefficients, and print its test accuracy and coefficients.

In [3]:
X_train, X_test = network.loc[train_rows, FEATURES], network.loc[test_rows, FEATURES]
y_train, y_test = snow[train_rows], snow[test_rows]
scaler = StandardScaler().fit(X_train)

weak = LogisticRegression(C=0.01).fit(scaler.transform(X_train), y_train)
print("test accuracy:", round(weak.score(scaler.transform(X_test), y_test), 3))
print(pd.Series(weak.coef_[0], index=FEATURES).round(2))

test accuracy: 0.83
latitude       0.61
longitude      0.55
elevation_m    0.49
coast_km       0.45
installed     -0.02
dtype: float64


A small `C` penalizes large coefficients more, so every coefficient shrinks toward 0. The penalty treats every column alike, which is fair only when the columns are on one scale.

**3.** Encode `owner` alone with `OneHotEncoder(sparse_output=False)`, fitted on the training rows, and print how many test stations each owner column marks with a 1.

In [4]:
owner_encoder = OneHotEncoder(sparse_output=False).fit(network.loc[train_rows, ["owner"]])
marks = owner_encoder.transform(network.loc[test_rows, ["owner"]])

print(pd.Series(marks.sum(axis=0), index=owner_encoder.get_feature_names_out()))

owner_airport     32.0
owner_met        131.0
owner_private     50.0
owner_road        87.0
dtype: float64


Every row has exactly one 1, so the four counts add up to the 300 test stations.

**4.** Train a `DecisionTreeClassifier(max_depth=4, random_state=0)` on the combined scaled and encoded columns, and print its test accuracy.

In [5]:
encoder = OneHotEncoder(sparse_output=False).fit(network.loc[train_rows, TEXT])
train_text = encoder.transform(network.loc[train_rows, TEXT])
test_text = encoder.transform(network.loc[test_rows, TEXT])
both_train = np.hstack([scaler.transform(X_train), train_text])
both_test = np.hstack([scaler.transform(X_test), test_text])

tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(both_train, y_train)
print(round(tree.score(both_test, y_test), 3))

0.827


The tree scores 0.827, as it did on the numeric columns alone, while the logistic regression in the worked examples scored 0.86 with terrain and owner and 0.84 without them on this split.

---

&#8592; **Back to:** [Preprocessing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/04-preprocessing.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
